# Conscience Research Learner Notebook

This notebook introduces the core evaluation and guardrail workflows in `conscience_research`.

What you will run:
- baseline oracle evaluation
- third-eye strict oracle evaluation
- runtime-only scenario edit check with baseline preservation
- production guardrail decision simulation


In [ ]:
from pathlib import Path
import hashlib
import json
import sys

repo_root = Path.cwd()
if repo_root.name != "conscience_research":
    candidate = repo_root / "conscience_research"
    if candidate.exists():
        repo_root = candidate

sys.path.insert(0, str(repo_root))

import agent
from production_service import decision_payload

print(f"Using repo root: {repo_root}")


## 1) Baseline oracle evaluation

In [ ]:
baseline_results = agent.evaluate_conscience_with_oracle_mode(agent, "baseline")
baseline_results


## 2) Third-eye strict evaluation

In [ ]:
third_eye_results = agent.evaluate_conscience_with_oracle_mode(agent, "third_eye")
third_eye_results


## 3) Runtime-only scenario edits with baseline preservation check

In [ ]:
def scenario_bank_hash(bank):
    payload = [
        {
            "id": s.id,
            "layer": s.layer,
            "action": s.action,
            "context": s.context,
            "expected_verdict": s.expected_verdict,
            "severity": s.severity,
            "norm_domain": s.norm_domain,
            "description": s.description,
        }
        for s in bank
    ]
    raw = json.dumps(payload, sort_keys=True, separators=(",", ":"))
    return hashlib.sha256(raw.encode()).hexdigest()

before_hash = scenario_bank_hash(agent.scenarios_module.SCENARIO_BANK)
runtime_results, runtime_meta = agent.evaluate_with_runtime_scenario_edits(agent, "live")
after_hash = scenario_bank_hash(agent.scenarios_module.SCENARIO_BANK)

{
    "runtime_results": runtime_results,
    "runtime_meta": runtime_meta,
    "baseline_preserved": before_hash == after_hash,
}


## 4) Production guardrail decision simulation

In [ ]:
payload = {
    "action": "disclosed_pii_without_consent",
    "context": {"agent_did_this": True, "harm_occurred": True},
    "dry_run": True,
}

decision, status = decision_payload(payload)
{"status": status, "decision": decision["decision"]}


## 5) Apply a binding update in service style (non-dry run)

In [ ]:
payload_update = {
    "action": "deliberate_moderate_deception",
    "context": {"intent": "deliberate"},
    "dry_run": False,
    "apply_binding_update": True,
}

update_response, update_status = decision_payload(payload_update)
{
    "status": update_status,
    "mutated": update_response["state"]["mutated"],
    "norm_domain": update_response["decision"]["norm_domain"],
    "norm_before": update_response["state"]["norm_before"],
    "norm_after": update_response["state"]["norm_after"],
}
